Passo 1: conferir o global pela Silver

In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.process_sigma.silver_orders")

n = silver.count()
late = silver.filter("is_late").count()

otd = round((n - late) / n * 100, 2)
dpmo = round(late / n * 1_000_000, 2)

print(f"Pedidos: {n} | Atrasados: {late}")
print(f"OTD: {otd}% | DPMO: {dpmo:,.0f}")

Pedidos: 500 | Atrasados: 260
OTD: 48.0% | DPMO: 520,000


Passo 2: a prova do σ e a coluna teórica

In [0]:
import math
from statistics import NormalDist

def dpmo_to_sigma(dpmo):  # igual ao notebook original
    if dpmo <= 0:       return 6.0
    if dpmo >= 999999:  return 0.5
    if dpmo >= 500000:  return 1.5
    if dpmo >= 308538:  return 2.0
    if dpmo >= 66807:   return 3.0
    if dpmo >= 6210:    return 4.0
    if dpmo >= 233:     return 5.0
    x = 29.37 - 2.221 * math.log(dpmo)
    return round(0.8406 + math.sqrt(max(0, x)), 2) if x > 0 else 0.5

nd = NormalDist()
def sigma_teorico(dpmo):
    if dpmo <= 0:
        return 6.0
    return round(nd.inv_cdf(1 - dpmo / 1_000_000) + 1.5, 2)

g = spark.table("workspace.process_sigma.gold_sigma").orderBy("period").toPandas()
g["sigma_python_original"] = g["dpmo"].apply(dpmo_to_sigma)
g["bate"] = g["sigma_python_original"] == g["sigma_level"]
g["sigma_teorico"] = g["dpmo"].apply(sigma_teorico)
g["diferenca"] = (g["sigma_level"] - g["sigma_teorico"]).round(2)

print("Global (original, teórico):", dpmo_to_sigma(520000.0), sigma_teorico(520000.0))
display(g[["period", "dpmo", "sigma_level", "sigma_python_original", "bate", "sigma_teorico", "diferenca"]])

Global (original, teórico): 1.5 1.45


period,dpmo,sigma_level,sigma_python_original,bate,sigma_teorico,diferenca
2022-01,500000.0,1.5,1.5,true,1.5,0.0
2022-02,482142.86,2.0,2.0,true,1.54,0.46
2022-03,580645.16,1.5,1.5,true,1.3,0.2
2022-04,483333.33,2.0,2.0,true,1.54,0.46
2022-05,500000.0,1.5,1.5,true,1.5,0.0
2022-06,516666.67,1.5,1.5,true,1.46,0.04
2022-07,532258.06,1.5,1.5,true,1.42,0.08
2022-08,548387.1,1.5,1.5,true,1.38,0.12
2022-09,571428.57,1.5,1.5,true,1.32,0.18


Passo 3: guardar a comparação e checar os status

In [0]:
from pyspark.sql import functions as F

# 3.1 Guardar a comparação como tabela Delta
(
    spark.createDataFrame(g[["period", "dpmo", "sigma_level", "sigma_teorico", "diferenca"]])
    .write.mode("overwrite")
    .saveAsTable("workspace.process_sigma.gold_sigma_comparativo")
)

# 3.2 O status em texto concorda com a regra dos dias?
silver = spark.table("workspace.process_sigma.silver_orders")

print("Valores do delivery_status:")
silver.groupBy("delivery_status").count().show()

print("Status 'Late delivery' x is_late (regra dos dias):")
(
    silver
    .withColumn("status_diz_atraso", F.col("delivery_status") == "Late delivery")
    .groupBy("status_diz_atraso", "is_late").count()
    .orderBy("status_diz_atraso", "is_late")
    .show()
)

print("late_delivery_risk x is_late:")
silver.groupBy("late_delivery_risk", "is_late").count().orderBy("late_delivery_risk", "is_late").show()

Valores do delivery_status:
+----------------+-----+
| delivery_status|count|
+----------------+-----+
|   Late delivery|  270|
|Shipping on time|  128|
|Advance shipping|  102|
+----------------+-----+

Status 'Late delivery' x is_late (regra dos dias):
+-----------------+-------+-----+
|status_diz_atraso|is_late|count|
+-----------------+-------+-----+
|            false|  false|  116|
|            false|   true|  114|
|             true|  false|  124|
|             true|   true|  146|
+-----------------+-------+-----+

late_delivery_risk x is_late:
+------------------+-------+-----+
|late_delivery_risk|is_late|count|
+------------------+-------+-----+
|                 0|  false|  117|
|                 0|   true|  134|
|                 1|  false|  123|
|                 1|   true|  126|
+------------------+-------+-----+

